In [1]:
from instruments import EuropeanOption
from bsm_pricer import BSMEuropeanPricer
from display import display_pricer_result

# Test option pricer

In [2]:
option = EuropeanOption(
    S=190,
    K=200,
    T=30 / 365,
    r=0.045,
    q=0.005,
    sigma=0.25,
    option_type="call"
)

pricer = BSMEuropeanPricer(option)

display_pricer_result(pricer)

,Value
Model,BSMEuropeanPricer
Option Type,call
Spot,190
Strike,200
Maturity,0.082192
Rate,0.045
Dividend Yield,0.005
Volatility,0.25
Price,2.095329
Delta,0.262948


In [3]:
S0 = 159.35000610351562
K = 150
T = 0.25
r = 0.045
q = 0.035
sigma = 0.25

put_option = EuropeanOption(
    S=S0,
    K=K,
    T=T,
    r=r,
    q=q,
    sigma=sigma,
    option_type="put"
)

put_pricer = BSMEuropeanPricer(put_option)

display_pricer_result(put_pricer)

,Value
Model,BSMEuropeanPricer
Option Type,put
Spot,159.350006
Strike,150
Maturity,0.25
Rate,0.045
Dividend Yield,0.035
Volatility,0.25
Price,3.758693
Delta,-0.283126


In [ ]:
# deep ITM call option, simulate buffer ETFs first layer.
display_pricer_result(BSMEuropeanPricer(
    EuropeanOption(
        S=10000, 
        K=100, 
        T=0.5, 
        r=0.05, 
        q=0.0, 
        sigma=0.2, 
        option_type="call"
        )
)
)


,Value
Model,BSMEuropeanPricer
Option Type,call
Spot,10000
Strike,100
Maturity,0.5
Rate,0.05
Dividend Yield,0.0
Volatility,0.2
Price,9902.469009
Delta,1.0


# Test futures pricer

In [13]:
from futures_pricer import price_futures

F = price_futures(S=100, T=1, r=0.05, c=0.0, q=0.01)
print(F)

104.08107741923882


# price 100% buffer ETF

In [22]:
S = 100
K_H = 90
K_cap = 110
T = 1
r = 0.05
q = 0.01
sigma = 0.2


put = EuropeanOption(
    S=S,
    K=K_H,
    T=T,
    r=r,
    q=q,
    sigma=sigma,
    option_type="put"
)

call_cap = EuropeanOption(
    S=S,
    K=K_cap,
    T=T,
    r=r,
    q=q,
    sigma=sigma,
    option_type="call"
)


stock_value = S
put_price = BSMEuropeanPricer(put).price()
call_cap_price = BSMEuropeanPricer(call_cap).price()

combo_price = stock_value + put_price - call_cap_price

# print("Stock value:", stock_value)
# print("Long put price:", put_price)
# print("Short call value:", -call_cap_price)
print("Combo price:", combo_price)


Combo price: 96.90380237015637


In [28]:
import numpy as np
from scipy.optimize import brentq

# ── Parameters ──────────────────────────────────────
S = 100
K_H = 100   # top of buffer = spot (ATM put, full protection)
T = 1
r = 0.05
q = 0.01
sigma = 0.20

# ── Leg 1: Hold underlying ───────────────────────────
price_underlying = S * np.exp(-q * T)

# ── Leg 2: BUY put @ K_H (ATM, full cover) ──────────
put_H = EuropeanOption(S=S, K=K_H, T=T, r=r, q=q, sigma=sigma, option_type="put")
price_put_H = BSMEuropeanPricer(put_H).price()

# ── Option cost to cover ─────────────────────────────
option_cost = price_put_H
print(f"put_H price (ATM):     {price_put_H:.4f}")
print(f"Option cost to cover:  {option_cost:.4f}")

# ── Diagnose range ───────────────────────────────────
def net_cost(k_cap):
    call = EuropeanOption(S=S, K=k_cap, T=T, r=r, q=q, sigma=sigma, option_type="call")
    return option_cost - BSMEuropeanPricer(call).price()

print(f"\nDiagnose:")
print(f"  net_cost(S)     = {net_cost(S):.4f}")
print(f"  net_cost(S*2.0) = {net_cost(S*2.0):.4f}")

# ── Solve for zero-cost K_cap ────────────────────────
K_cap_zerocost = brentq(net_cost, S, S * 2.0)

call_cap = EuropeanOption(S=S, K=K_cap_zerocost, T=T, r=r, q=q, sigma=sigma, option_type="call")
price_call_cap = BSMEuropeanPricer(call_cap).price()

print(f"\n{'='*45}")
print(f"put_H  (buy, ATM):      +{price_put_H:.4f}")
print(f"call_cap (sell):        -{price_call_cap:.4f}")
print(f"Net option cost:         {option_cost - price_call_cap:.6f}  <- ≈ 0")
print(f"\nZero-cost K_cap:         {K_cap_zerocost:.4f}")
print(f"Upside cap vs spot:      {(K_cap_zerocost/S - 1)*100:.2f}%")
print(f"Buffer:                  100%")
print(f"{'='*45}")

put_H price (ATM):     5.9443
Option cost to cover:  5.9443

Diagnose:
  net_cost(S)     = -3.8820
  net_cost(S*2.0) = 5.9403

put_H  (buy, ATM):      +5.9443
call_cap (sell):        -5.9443
Net option cost:         0.000000  <- ≈ 0

Zero-cost K_cap:         109.0067
Upside cap vs spot:      9.01%
Buffer:                  100%


# Price 15% buffer ETF

In [26]:
import numpy as np
from scipy.optimize import brentq

# ── Parameters ──────────────────────────────────────
S = 100
K_H = 100   # top of buffer (= spot, ATM)
K_L = 85    # 15% buffer below K_H
T = 1
r = 0.05
q = 0.01
sigma = 0.20

# ── Leg 2: BUY put @ K_H ────────────────────────────
put_H = EuropeanOption(S=S, K=K_H, T=T, r=r, q=q, sigma=sigma, option_type="put")
price_put_H = BSMEuropeanPricer(put_H).price()

# ── Leg 3: SELL put @ K_L ───────────────────────────
put_L = EuropeanOption(S=S, K=K_L, T=T, r=r, q=q, sigma=sigma, option_type="put")
price_put_L = BSMEuropeanPricer(put_L).price()

# ── Option-only net cost ─────────────────────────────
# Zero-cost condition: call_cap premium = put_H - put_L
option_cost = price_put_H - price_put_L
print(f"put_H price:           {price_put_H:.4f}")
print(f"put_L price:           {price_put_L:.4f}")
print(f"Option cost to cover:  {option_cost:.4f}  <- call_cap must generate this")

# ── Solve for zero-cost K_cap ────────────────────────
def net_cost(k_cap):
    call = EuropeanOption(S=S, K=k_cap, T=T, r=r, q=q, sigma=sigma, option_type="call")
    return option_cost - BSMEuropeanPricer(call).price()

print(f"\nDiagnose:")
print(f"  net_cost(S)     = {net_cost(S):.4f}    <- should be > 0")
print(f"  net_cost(S*2.0) = {net_cost(S*2.0):.4f}  <- should be < 0")

K_cap_zerocost = brentq(net_cost, S, S * 2.0)

call_cap = EuropeanOption(S=S, K=K_cap_zerocost, T=T, r=r, q=q, sigma=sigma, option_type="call")
price_call_cap = BSMEuropeanPricer(call_cap).price()

print(f"\n{'='*45}")
print(f"put_H  (buy):          +{price_put_H:.4f}")
print(f"put_L  (sell):         -{price_put_L:.4f}")
print(f"call_cap (sell):       -{price_call_cap:.4f}")
print(f"Net option cost:        {option_cost - price_call_cap:.6f}  <- ≈ 0")
print(f"\nZero-cost K_cap:        {K_cap_zerocost:.4f}")
print(f"Upside cap vs spot:     {(K_cap_zerocost/S - 1)*100:.2f}%")
print(f"Buffer spread:          {(K_H - K_L)/K_H*100:.1f}%")
print(f"{'='*45}")

put_H price:           5.9443
put_L price:           1.4508
Option cost to cover:  4.4934  <- call_cap must generate this

Diagnose:
  net_cost(S)     = -5.3329    <- should be > 0
  net_cost(S*2.0) = 4.4895  <- should be < 0

put_H  (buy):          +5.9443
put_L  (sell):         -1.4508
call_cap (sell):       -4.4934
Net option cost:        0.000000  <- ≈ 0

Zero-cost K_cap:        113.6003
Upside cap vs spot:     13.60%
Buffer spread:          15.0%


# Show how to use parameter tools

In [5]:
from parameter_tools import get_stock_price

In [29]:
# if i want to today's stock price
price=get_stock_price("XOM",known_price=None, price_date="today")
price

{'date': '2026-05-08',
 'open': 145.85000610351562,
 'high': 146.5,
 'low': 144.19500732421875,
 'close': 144.4550018310547}

In [7]:
price=get_stock_price("XOM",known_price=None, price_date="2026-02-28")
price

$XOM: possibly delisted; no price data found  (1d 2026-02-28 -> 2026-03-01)


{'requested_date': '2026-02-28',
 'previous': {'date': '2026-02-27',
  'open': 151.0,
  'high': 153.64999389648438,
  'low': 149.25,
  'close': 152.5},
 'next': {'date': '2026-03-02',
  'open': 159.35000610351562,
  'high': 159.61000061035156,
  'low': 153.02999877929688,
  'close': 154.22000122070312}}

In [8]:
from parameter_tools import get_time_to_maturity

In [9]:
get_time_to_maturity(
    expiry_date="2026-06-19",
    from_date="2024-01-01",
    day_count="calendar"
)


{'days': 900, 'years': 2.4657534246575343, 'count_type': 'calendar'}

In [11]:
get_time_to_maturity(
    "2026-06-19", 
    from_date="today", 
    day_count="business"
    )


{'days': 30, 'years': 0.11904761904761904, 'count_type': 'business'}

In [12]:
get_time_to_maturity(
    "2026-06-19", 
    from_date="today", 
    day_count="calendar")


{'days': 44, 'years': 0.12054794520547946, 'count_type': 'calendar'}